# LLM-SATD Alt-Test Metrics

This notebook generates `alt_test_metrics.csv` from `alt_test_labels.csv`, including Cohen's kappa and Gwet's AC1 agreement metrics.

The Alt-Test implementation below follows the reference implementation distributed in `AltTest/alt_test_example.ipynb`.

In [49]:
from __future__ import annotations

import csv
import math
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence

import numpy as np
from scipy.stats import ttest_1samp
from sklearn.metrics import cohen_kappa_score

NOTEBOOK_DIR = Path("classify_validation")
if not (NOTEBOOK_DIR / "alt_test_labels.csv").exists():
    NOTEBOOK_DIR = Path(".")

INPUT_CSV = NOTEBOOK_DIR / "alt_test_labels.csv"
ALT_TEST_METRICS_CSV = NOTEBOOK_DIR / "alt_test_metrics.csv"
VALIDATION_LABELS_CSV = NOTEBOOK_DIR / "llm_validate_labels.csv"
VALIDATION_KAPPA_CSV = NOTEBOOK_DIR / "llm_validate_kappa.csv"
HUMAN_COLUMNS = (
    "annotator1_is_llm_satd",
    "annotator2_is_llm_satd",
    "annotator3_is_llm_satd",
)
MODEL_COLUMNS = ("gpt4o_is_llm_satd",)
VALIDATION_MODEL_COLUMN = "gpt4o_is_llm_satd"
VALIDATION_HUMAN_COLUMN = "annotator1_is_llm_satd"
VALIDATION_MODEL_NAME = "gpt4o"
VALIDATION_HUMAN_NAME = "annotator1"
CONSENSUS_COLUMN = "consensus"
EPSILON = 0.15
Q_FDR = 0.05
MIN_HUMANS_PER_INSTANCE = 3
MIN_INSTANCES_PER_HUMAN = 100

METRIC_FIELDNAMES = [
    "metric_group",
    "comparison",
    "left_labeler",
    "right_labeler",
    "model",
    "n_items",
    "value",
    "passes",
    "tp",
    "tn",
    "fp",
    "fn",
    "precision",
    "recall",
    "f1",
    "accuracy",
    "positive_rate",
]

TABLE_FLOAT_FIELDS = {
    "value",
    "precision",
    "recall",
    "f1",
    "accuracy",
    "positive_rate",
    "cohen_kappa",
    "gwet_ac1",
    "observed_agreement",
    "expected_agreement",
    "left_positive_rate",
    "right_positive_rate",
}

## Alt-Test Implementation

The function below mirrors the reference notebook's leave-one-annotator-out test for accuracy labels.

In [50]:
def by_procedure(p_values: Sequence[float], q: float) -> List[int]:
    values = np.array(p_values, dtype=float)
    if values.size == 0:
        return []
    m = len(values)
    sorted_indices = np.argsort(values)
    sorted_pvals = values[sorted_indices]
    harmonic_m = np.sum(1.0 / np.arange(1, m + 1))
    thresholds = (np.arange(1, m + 1) / m) * (q / harmonic_m)
    max_i = -1
    for index, p_value in enumerate(sorted_pvals):
        if p_value <= thresholds[index]:
            max_i = index
    if max_i == -1:
        return []
    return list(sorted_indices[: max_i + 1])


def accuracy(prediction: Any, annotations: Sequence[Any]) -> float:
    return float(np.mean([prediction == annotation for annotation in annotations]))


def alt_test(
    llm_annotations: Dict[str, Any],
    humans_annotations: Dict[str, Dict[str, Any]],
    scoring_function: str | Callable[[Any, Sequence[Any]], float] = "accuracy",
    epsilon: float = 0.2,
    q_fdr: float = 0.05,
    min_humans_per_instance: int = 2,
    min_instances_per_human: int = 30,
) -> tuple[float, float]:
    if isinstance(scoring_function, str):
        if scoring_function != "accuracy":
            raise ValueError("Only accuracy scoring is used for this binary validation task.")
        scorer = accuracy
    else:
        scorer = scoring_function

    annotator_items: Dict[str, List[str]] = {}
    instance_annotators: Dict[str, List[str]] = {}
    for annotator, annotations in humans_annotations.items():
        annotator_items[annotator] = list(annotations.keys())
        for item_id in annotations:
            instance_annotators.setdefault(item_id, []).append(annotator)

    instances_to_keep = {
        item_id
        for item_id, annotators in instance_annotators.items()
        if len(annotators) >= min_humans_per_instance and item_id in llm_annotations
    }
    annotator_items = {
        annotator: [item_id for item_id in item_ids if item_id in instances_to_keep]
        for annotator, item_ids in annotator_items.items()
    }
    instance_annotators = {
        item_id: annotators
        for item_id, annotators in instance_annotators.items()
        if item_id in instances_to_keep
    }

    p_values: List[float] = []
    advantage_probs: List[float] = []
    humans: List[str] = []
    for excluded_annotator in humans_annotations:
        llm_indicators: List[int] = []
        excluded_indicators: List[int] = []
        instances = [
            item_id
            for item_id in annotator_items[excluded_annotator]
            if item_id in llm_annotations
        ]
        if len(instances) < min_instances_per_human:
            continue

        for item_id in instances:
            human_annotation = humans_annotations[excluded_annotator][item_id]
            llm_annotation = llm_annotations[item_id]
            remaining_annotations = [
                humans_annotations[annotator][item_id]
                for annotator in instance_annotators[item_id]
                if annotator != excluded_annotator
            ]
            human_score = scorer(human_annotation, remaining_annotations)
            llm_score = scorer(llm_annotation, remaining_annotations)
            llm_indicators.append(1 if llm_score >= human_score else 0)
            excluded_indicators.append(1 if human_score >= llm_score else 0)

        diff_indicators = [
            excluded_value - llm_value
            for excluded_value, llm_value in zip(excluded_indicators, llm_indicators)
        ]
        p_values.append(float(ttest_1samp(diff_indicators, epsilon, alternative="less").pvalue))
        advantage_probs.append(float(np.mean(llm_indicators)))
        humans.append(excluded_annotator)

    rejected_indices = by_procedure(p_values, q_fdr)
    advantage_probability = float(np.mean(advantage_probs)) if advantage_probs else math.nan
    winning_rate = len(rejected_indices) / len(humans) if humans else math.nan
    return winning_rate, advantage_probability

## Metrics Helpers

In [51]:
def parse_optional_bool(value: object) -> Optional[bool]:
    if value is None:
        return None
    text = str(value).strip().lower()
    if text == "":
        return None
    if text == "true":
        return True
    if text == "false":
        return False
    raise ValueError(f"Cannot parse boolean value: {value!r}")


def finite_or_none(value: float) -> Optional[float]:
    if math.isnan(value) or math.isinf(value):
        return None
    return value


def row_identifier(row: Dict[str, str], index: int) -> str:
    full_name = str(row.get("full_name", "")).strip()
    id_in_repo = str(row.get("id_in_repo", "")).strip()
    if full_name and id_in_repo:
        return f"{full_name}::{id_in_repo}"
    return str(index)


def bool_annotations(rows: Sequence[Dict[str, str]], column: str) -> Dict[str, bool]:
    annotations: Dict[str, bool] = {}
    for index, row in enumerate(rows):
        value = parse_optional_bool(row.get(column))
        if value is not None:
            annotations[row_identifier(row, index)] = value
    return annotations


def column_values(rows: Sequence[Dict[str, str]], column: str) -> List[Optional[bool]]:
    return [parse_optional_bool(row.get(column)) for row in rows]


def positive_rate(values: Iterable[Optional[bool]]) -> Optional[float]:
    observed = [value for value in values if value is not None]
    if not observed:
        return None
    return sum(observed) / len(observed)


def labeler_name(column: str) -> str:
    suffix = "_is_llm_satd"
    return column[: -len(suffix)] if column.endswith(suffix) else column


def metric_row(**values: object) -> Dict[str, object]:
    return {field: values.get(field, "") for field in METRIC_FIELDNAMES}


def format_table_value(field: str, value: object) -> object:
    if value is None or value == "":
        return ""
    if field not in TABLE_FLOAT_FIELDS:
        return value
    try:
        numeric_value = float(value)
    except (TypeError, ValueError):
        return value
    if not math.isfinite(numeric_value):
        return ""
    return f"{numeric_value:.2f}"


def format_table_row(row: Dict[str, object], fieldnames: Sequence[str]) -> Dict[str, object]:
    return {field: format_table_value(field, row.get(field, "")) for field in fieldnames}


def observed_agreement(left_values: Sequence[Optional[bool]], right_values: Sequence[Optional[bool]]) -> tuple[Optional[float], int]:
    pairs = [
        (left, right)
        for left, right in zip(left_values, right_values)
        if left is not None and right is not None
    ]
    if not pairs:
        return None, 0
    return sum(left == right for left, right in pairs) / len(pairs), len(pairs)


def cohen_kappa(left_values: Sequence[Optional[bool]], right_values: Sequence[Optional[bool]]) -> tuple[Optional[float], int]:
    pairs = [
        (left, right)
        for left, right in zip(left_values, right_values)
        if left is not None and right is not None
    ]
    if not pairs:
        return None, 0
    left, right = zip(*pairs)
    return finite_or_none(float(cohen_kappa_score(left, right))), len(pairs)


def krippendorff_alpha_nominal(units: Sequence[Sequence[Optional[bool]]]) -> tuple[Optional[float], int]:
    pairable_units = []
    category_totals = {False: 0, True: 0}
    total_annotations = 0
    observed_disagreement = 0.0

    for unit in units:
        values = [value for value in unit if value is not None]
        if len(values) < 2:
            continue
        pairable_units.append(values)
        n_values = len(values)
        counts = {False: values.count(False), True: values.count(True)}
        for category, count in counts.items():
            category_totals[category] += count
        total_annotations += n_values
        observed_disagreement += (
            n_values * n_values - sum(count * count for count in counts.values())
        ) / (n_values - 1)

    if not pairable_units or total_annotations < 2:
        return None, 0

    expected_disagreement = (
        total_annotations * total_annotations
        - sum(count * count for count in category_totals.values())
    ) / (total_annotations - 1)
    if expected_disagreement == 0:
        return None, len(pairable_units)
    alpha = 1 - (observed_disagreement / expected_disagreement)
    return finite_or_none(float(alpha)), len(pairable_units)


def pairwise_krippendorff_alpha(left_values: Sequence[Optional[bool]], right_values: Sequence[Optional[bool]]) -> tuple[Optional[float], int]:
    return krippendorff_alpha_nominal([[left, right] for left, right in zip(left_values, right_values)])


def gwet_ac1_nominal(units: Sequence[Sequence[Optional[bool]]]) -> tuple[Optional[float], int]:
    pairable_units = []
    category_totals = {False: 0, True: 0}
    total_annotations = 0
    observed_agreements: List[float] = []

    for unit in units:
        values = [value for value in unit if value is not None]
        if len(values) < 2:
            continue
        pairable_units.append(values)
        n_values = len(values)
        counts = {False: values.count(False), True: values.count(True)}
        for category, count in counts.items():
            category_totals[category] += count
        total_annotations += n_values
        observed_agreements.append(
            sum(count * (count - 1) for count in counts.values()) / (n_values * (n_values - 1))
        )

    if not pairable_units or total_annotations < 2:
        return None, 0

    observed_agreement = float(np.mean(observed_agreements))
    category_proportions = [count / total_annotations for count in category_totals.values()]
    chance_agreement = sum(p * (1 - p) for p in category_proportions)
    denominator = 1 - chance_agreement
    if denominator == 0:
        return None, len(pairable_units)
    ac1 = (observed_agreement - chance_agreement) / denominator
    return finite_or_none(float(ac1)), len(pairable_units)


def pairwise_gwet_ac1(left_values: Sequence[Optional[bool]], right_values: Sequence[Optional[bool]]) -> tuple[Optional[float], int]:
    return gwet_ac1_nominal([[left, right] for left, right in zip(left_values, right_values)])


In [52]:
def majority_labels(rows: Sequence[Dict[str, str]], human_columns: Sequence[str]) -> List[Optional[bool]]:
    labels: List[Optional[bool]] = []
    for row in rows:
        observed = []
        for column in human_columns:
            value = parse_optional_bool(row.get(column))
            if value is not None:
                observed.append(value)
        if len(observed) < 2:
            labels.append(None)
            continue
        true_count = sum(observed)
        false_count = len(observed) - true_count
        labels.append(None if true_count == false_count else true_count > false_count)
    return labels


def reference_labels(rows: Sequence[Dict[str, str]]) -> tuple[str, List[Optional[bool]]]:
    if not rows or CONSENSUS_COLUMN not in rows[0]:
        raise ValueError(f"Missing required column: {CONSENSUS_COLUMN}")
    consensus = column_values(rows, CONSENSUS_COLUMN)
    if not any(value is not None for value in consensus):
        raise ValueError("Consensus column is present but contains no usable labels.")
    return "consensus", consensus


def pairwise_metric_rows(
    metric_group: str,
    comparison_prefix: str,
    left_column: str,
    right_column: str,
    left_values: Sequence[Optional[bool]],
    right_values: Sequence[Optional[bool]],
    model_column: Optional[str] = None,
) -> List[Dict[str, object]]:
    kappa, n_items = cohen_kappa(left_values, right_values)
    ac1, ac1_n_items = pairwise_gwet_ac1(left_values, right_values)
    left_name = labeler_name(left_column)
    right_name = labeler_name(right_column)
    model_name = labeler_name(model_column) if model_column else ""
    rate = positive_rate(left_values)
    return [
        metric_row(
            metric_group=metric_group,
            comparison=f"cohen_kappa_{comparison_prefix}",
            left_labeler=left_name,
            right_labeler=right_name,
            model=model_name,
            n_items=n_items,
            value=kappa,
            positive_rate=rate,
        ),
        metric_row(
            metric_group=metric_group,
            comparison=f"gwet_ac1_{comparison_prefix}",
            left_labeler=left_name,
            right_labeler=right_name,
            model=model_name,
            n_items=ac1_n_items,
            value=ac1,
            positive_rate=rate,
        ),
    ]


def classification_metrics(
    predictions: Sequence[Optional[bool]],
    truths: Sequence[Optional[bool]],
) -> Dict[str, Optional[float]]:
    pairs = [
        (prediction, truth)
        for prediction, truth in zip(predictions, truths)
        if prediction is not None and truth is not None
    ]
    if not pairs:
        return {
            "n_items": 0,
            "tp": 0,
            "tn": 0,
            "fp": 0,
            "fn": 0,
            "precision": None,
            "recall": None,
            "f1": None,
            "accuracy": None,
        }
    tp = sum(1 for prediction, truth in pairs if prediction and truth)
    tn = sum(1 for prediction, truth in pairs if (not prediction) and (not truth))
    fp = sum(1 for prediction, truth in pairs if prediction and (not truth))
    fn = sum(1 for prediction, truth in pairs if (not prediction) and truth)
    precision = tp / (tp + fp) if (tp + fp) else None
    recall = tp / (tp + fn) if (tp + fn) else None
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision is not None and recall is not None and (precision + recall)
        else None
    )
    accuracy_value = (tp + tn) / len(pairs)
    return {
        "n_items": len(pairs),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy_value,
    }

## Generate `alt_test_metrics.csv`

In [53]:
with INPUT_CSV.open(newline="", encoding="utf-8") as handle:
    alt_test_rows = list(csv.DictReader(handle))

fieldnames = set(alt_test_rows[0]) if alt_test_rows else set()
missing_columns = [column for column in [*HUMAN_COLUMNS, *MODEL_COLUMNS] if column not in fieldnames]
if missing_columns:
    raise ValueError(f"Missing required columns: {', '.join(missing_columns)}")

human_annotation_map = {
    column: bool_annotations(alt_test_rows, column)
    for column in HUMAN_COLUMNS
}
metrics: List[Dict[str, object]] = []

for model_column in MODEL_COLUMNS:
    model_annotations = bool_annotations(alt_test_rows, model_column)
    winning_rate, advantage_probability = alt_test(
        llm_annotations=model_annotations,
        humans_annotations=human_annotation_map,
        scoring_function="accuracy",
        epsilon=EPSILON,
        q_fdr=Q_FDR,
        min_humans_per_instance=MIN_HUMANS_PER_INSTANCE,
        min_instances_per_human=MIN_INSTANCES_PER_HUMAN,
    )
    model_name = labeler_name(model_column)
    passes = "yes" if winning_rate >= 0.5 else "no"
    metrics.append(
        metric_row(
            metric_group="alt_test",
            comparison="winning_rate",
            model=model_name,
            n_items=len(model_annotations),
            value=finite_or_none(winning_rate),
            passes=passes,
        )
    )
    metrics.append(
        metric_row(
            metric_group="alt_test",
            comparison="advantage_probability",
            model=model_name,
            n_items=len(model_annotations),
            value=finite_or_none(advantage_probability),
            passes=passes,
        )
    )

for left_index, left_column in enumerate(HUMAN_COLUMNS):
    for right_column in HUMAN_COLUMNS[left_index + 1 :]:
        metrics.extend(
            pairwise_metric_rows(
                metric_group="annotator_pair_agreement",
                comparison_prefix="between_annotators",
                left_column=left_column,
                right_column=right_column,
                left_values=column_values(alt_test_rows, left_column),
                right_values=column_values(alt_test_rows, right_column),
            )
        )

overall_ac1, overall_ac1_n_items = gwet_ac1_nominal(
    [
        [parse_optional_bool(row.get(column)) for column in HUMAN_COLUMNS]
        for row in alt_test_rows
    ]
)
metrics.append(
    metric_row(
        metric_group="annotator_overall_agreement",
        comparison="gwet_ac1_all_annotators",
        n_items=overall_ac1_n_items,
        value=overall_ac1,
    )
)

reference_name, reference = reference_labels(alt_test_rows)
for column in [*HUMAN_COLUMNS, *MODEL_COLUMNS]:
    is_model = column in MODEL_COLUMNS
    metrics.extend(
        pairwise_metric_rows(
            metric_group=f"{reference_name}_agreement",
            comparison_prefix=f"with_{reference_name}",
            left_column=column,
            right_column=reference_name,
            left_values=column_values(alt_test_rows, column),
            right_values=reference,
            model_column=column if is_model else None,
        )
    )

for model_column in MODEL_COLUMNS:
    predictions = column_values(alt_test_rows, model_column)
    model_metrics = classification_metrics(predictions, reference)
    metrics.append(
        metric_row(
            metric_group=f"model_vs_{reference_name}",
            comparison="classification",
            model=labeler_name(model_column),
            n_items=model_metrics["n_items"],
            tp=model_metrics["tp"],
            tn=model_metrics["tn"],
            fp=model_metrics["fp"],
            fn=model_metrics["fn"],
            precision=model_metrics["precision"],
            recall=model_metrics["recall"],
            f1=model_metrics["f1"],
            accuracy=model_metrics["accuracy"],
            positive_rate=positive_rate(predictions),
        )
    )

with ALT_TEST_METRICS_CSV.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=METRIC_FIELDNAMES)
    writer.writeheader()
    writer.writerows(format_table_row(row, METRIC_FIELDNAMES) for row in metrics)

print(f"Wrote {ALT_TEST_METRICS_CSV} with {len(metrics)} metric rows")

Wrote alt_test_metrics.csv with 18 metric rows


## Generate `llm_validate_kappa.csv`

The full validation labels contain `annotator1_is_llm_satd` only; `annotator2_is_llm_satd` and `annotator3_is_llm_satd` are collected only for the 100-row Alt-Test subset in `alt_test_labels.csv`. This output includes Cohen's kappa and Gwet's AC1 for GPT-4o vs annotator1.


In [54]:
with VALIDATION_LABELS_CSV.open(newline="", encoding="utf-8") as handle:
    validation_rows = list(csv.DictReader(handle))

model_values = column_values(validation_rows, VALIDATION_MODEL_COLUMN)
human_values = column_values(validation_rows, VALIDATION_HUMAN_COLUMN)
pairs = [
    (model_value, human_value)
    for model_value, human_value in zip(model_values, human_values)
    if model_value is not None and human_value is not None
]
if not pairs:
    raise ValueError("No overlapping gpt4o/annotator1 labels found.")

predictions, truths = zip(*pairs)
kappa = float(cohen_kappa_score(predictions, truths))
ac1, _ = pairwise_gwet_ac1(list(predictions), list(truths))
raw_agreement = sum(prediction == truth for prediction, truth in pairs) / len(pairs)
left_positive_rate = sum(predictions) / len(predictions)
right_positive_rate = sum(truths) / len(truths)
expected_agreement = (left_positive_rate * right_positive_rate) + ((1 - left_positive_rate) * (1 - right_positive_rate))
model_metrics = classification_metrics(list(predictions), list(truths))

fieldnames = [
    "pair_id",
    "comparison_type",
    "left_model",
    "right_model",
    "cohen_kappa",
    "gwet_ac1",
    "observed_agreement",
    "expected_agreement",
    "n_items",
    "left_positive_rate",
    "right_positive_rate",
    "tp",
    "tn",
    "fp",
    "fn",
    "precision",
    "recall",
    "accuracy",
]
summary_row = {
    "pair_id": f"{VALIDATION_MODEL_NAME}_vs_{VALIDATION_HUMAN_NAME}",
    "comparison_type": "model_human",
    "left_model": VALIDATION_MODEL_NAME,
    "right_model": VALIDATION_HUMAN_NAME,
    "cohen_kappa": kappa,
    "gwet_ac1": ac1,
    "observed_agreement": raw_agreement,
    "expected_agreement": expected_agreement,
    "n_items": len(pairs),
    "left_positive_rate": left_positive_rate,
    "right_positive_rate": right_positive_rate,
    "tp": model_metrics["tp"],
    "tn": model_metrics["tn"],
    "fp": model_metrics["fp"],
    "fn": model_metrics["fn"],
    "precision": model_metrics["precision"],
    "recall": model_metrics["recall"],
    "accuracy": model_metrics["accuracy"],
}

with VALIDATION_KAPPA_CSV.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerow(format_table_row(summary_row, fieldnames))
print(f"Wrote {VALIDATION_KAPPA_CSV} with 1 metric row")


Wrote llm_validate_kappa.csv with 1 metric row
